# Initial-margin CVaR MKABSDE

This notebook displays the retained quadrature campaign, both proxy BSDEs, and the common-path comparison of three conditional-CVaR estimators.

## Complete reproduction commands

```bash
python -m initial_margin.run_campaign \
  --product all --steps 50 100 200 \
  --seeds 42 43 44 45 46 47 48 49 50 51 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.compare_risk_estimators \
  --product all --steps 200 --seeds 42 43 44 \
  --paths 200000 --degree 2 --device cuda

python -m initial_margin.deterministic_reference \
  --product all --steps 50 100 200 --space-steps 6000 \
  --expectation-nodes 24 \
  --output initial_margin/results/deterministic_reference.json
```

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').is_file():
    root = root.parent
if not (root / 'pyproject.toml').is_file():
    raise FileNotFoundError('repository root not found')
sys.path.insert(0, str(root))
from initial_margin.evaluation import ARTICLE_REFERENCE_VALUES

result_root = root / 'initial_margin' / 'results'
products = ('call', 'butterfly')
reference_payload = json.loads((result_root / 'deterministic_reference.json').read_text())
direct_references = {(row['product'], row['time_steps']): row for row in reference_payload['rows']}
method_payload = json.loads((result_root / 'cvar_method_comparison.json').read_text())

## Initial-margin-induced price corrections

Every number is $100\,(V_0-V_0^{BS})$. It is the price correction induced by funding initial margin, not the total price or the CVaR collateral.

In [2]:
for product in products:
    payload = json.loads((result_root / f'atm_{product}_experiment.json').read_text())
    base_price = payload['rows'][0]['metrics']['black_scholes_no_im']['price']
    article = ARTICLE_REFERENCE_VALUES[product]
    degree = payload['summary'][0]['hermite_degree']
    lines = [
        f'### ATM {product} — Hermite degree {degree}',
        '',
        '| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |',
        '|---:|---:|---:|---:|---:|',
    ]
    for row in payload['summary']:
        lines.append(
            f"| {row['time_steps']} | {100 * row['direct_mkabsde_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['direct_mkabsde_price_correction_vs_black_scholes_std']:.2e} | "
            f"{100 * direct_references[(product, row['time_steps'])]['price_correction']:.6f} | "
            f"{100 * row['nonlinear_proxy_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['nonlinear_proxy_price_correction_vs_black_scholes_std']:.2e} | "
            f"{100 * row['linear_proxy_price_correction_vs_black_scholes_mean']:.6f} ± {100 * row['linear_proxy_price_correction_vs_black_scholes_std']:.2e} |"
        )
    lines.append(
        f"| Published proxy value converted to correction | — | — | {100 * (article['nonlinear_proxy_price'] - base_price):.6f} | {100 * (article['linear_proxy_price'] - base_price):.6f} |"
    )
    display(Markdown('\n'.join(lines)))

### ATM call — Hermite degree 2

| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 1.873052 ± 2.85e-03 | 1.873235 | 2.197172 ± 2.87e-03 | 2.189706 ± 2.86e-03 |
| 100 | 1.868165 ± 1.97e-03 | 1.868720 | 2.190975 ± 2.03e-03 | 2.183479 ± 2.02e-03 |
| 200 | 1.866233 ± 2.38e-03 | 1.866000 | 2.188326 ± 2.47e-03 | 2.180814 ± 2.47e-03 |
| Published proxy value converted to correction | — | — | 2.178830 | 2.298830 |

### ATM butterfly — Hermite degree 2

| $N$ | direct MKABSDE | deterministic reference | nonlinear proxy | linear proxy |
|---:|---:|---:|---:|---:|
| 50 | 0.236180 ± 2.32e-04 | 0.236936 | 0.223964 ± 2.03e-04 | 0.223641 ± 2.03e-04 |
| 100 | 0.238494 ± 1.91e-04 | 0.239110 | 0.226279 ± 1.63e-04 | 0.225946 ± 1.63e-04 |
| 200 | 0.239310 ± 1.33e-04 | 0.239904 | 0.227199 ± 6.39e-05 | 0.226861 ± 6.41e-05 |
| Published proxy value converted to correction | — | — | 0.216529 | 0.216529 |

## Conditional-CVaR estimator audit

Only the CVaR engine changes below. The outer paths, Hermite regressions, and direct MKABSDE update are identical.

In [3]:
lines = [
    '| Product | CVaR estimator | $100(V_0-V_0^{BS})$ | seed std | GPU time | price difference vs quadrature |',
    '|---|---|---:|---:|---:|---:|',
]
for row in method_payload['summary']:
    lines.append(
        f"| {row['product']} | {row['method']} | {100 * row['im_correction_mean']:.6f} | "
        f"{100 * row['im_correction_std']:.2e} | {row['gpu_seconds_mean']:.2f} s | "
        f"{row['price_difference_vs_quadrature_mean']:+.2e} |"
    )
display(Markdown('\n'.join(lines)))

| Product | CVaR estimator | $100(V_0-V_0^{BS})$ | seed std | GPU time | price difference vs quadrature |
|---|---|---:|---:|---:|---:|
| butterfly | quadrature | 0.239215 | 8.04e-05 | 14.90 s | +0.00e+00 |
| butterfly | nested_mc | 0.240334 | 2.41e-03 | 60.10 s | +1.12e-05 |
| butterfly | learning | 0.242828 | 1.34e-03 | 51.94 s | +3.61e-05 |
| call | quadrature | 1.866591 | 3.24e-03 | 5.22 s | +0.00e+00 |
| call | nested_mc | 1.868394 | 2.63e-03 | 43.63 s | +1.80e-05 |
| call | learning | 1.871700 | 6.31e-03 | 51.16 s | +5.11e-05 |

## Interpretation

The three risk engines agree on the direct time-zero price to better than $10^{-4}$, so the direct/proxy gap is not caused by Gauss--Legendre quadrature. Quadrature is retained because it is deterministic conditional on the current regression and is the fastest method in dimension one. The deterministic state-space calculation is a separate same-grid regression check; it is not an exact continuous-time solution.